In [ ]:
import io
import json
import os
import time
from pathlib import Path
from urllib.parse import urlencode

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

try:
    from astropy.time import Time
except ImportError:
    Time = None

ZTF_LC_API_URL = "https://irsa.ipac.caltech.edu/cgi-bin/ZTF/nph_light_curves"
ZTF_TAP_URL = "https://irsa.ipac.caltech.edu/TAP/sync"
ZTF_OUTPUT_DIR = Path("ztf_dr")
ZTF_OUTPUT_DIR.mkdir(exist_ok=True)

# 5 arcsec, matching the Lasair cone-search radius used in periodicity_search.ipynb.
SEARCH_RADIUS_ARCSEC = 5.0
SEARCH_RADIUS_DEG = SEARCH_RADIUS_ARCSEC / 3600.0

# Exclude the common ZTF bad-data bit. Set to None if you want raw API output.
BAD_CATFLAGS_MASK = 32768

# Pin the public data release so the object lookup table and lightcurve collection match.
COLLECTION = "ztf_dr24"
ZTF_OBJECTS_TABLE = COLLECTION.replace("ztf_", "ztf_objects_")

ZTF_FP_CREDENTIAL_PATHS = 'ztf_fp_requests/.ztf_fp_credentials'
def get_ztffp_credentials():
    try:
        with open(ZTF_FP_CREDENTIAL_PATHS, 'r') as f:
            return f.read().strip().split('\n')
    except FileNotFoundError:
        return None

## Load the TNS Ib/c target table

In [56]:
ZTF_FP_MIN_JD = 2458194.5  # 2018-03-17

def load_tns_ibc():
    """Load the same TNS Ib/c sample as periodicity_search.ipynb when possible."""
    full_tns_path = Path("tns_public_objects_20250615.csv")

    if full_tns_path.exists():
        tns = pd.read_csv(full_tns_path, header=1)
        tns = tns[np.logical_and(tns["redshift"] <= 0.05, tns["declination"] > -60)]

        sn_typeids = [4, 5, 6, 7, 9, 106]  # SN Ib, Ic, Ib/c, Ic-BL, Ibn, Ia-CSM
        tns_ibc = tns[np.isin(tns["typeid"], sn_typeids)].copy()

        tns_ibc = tns_ibc[Time(pd.to_datetime(tns_ibc["discoverydate"],
                                              utc=True).dt.to_pydatetime()).jd > ZTF_FP_MIN_JD].copy()

        # exclude_names = ["2018ebt", "2018yn"]  # seem like CV?
        # tns_ibc = tns_ibc[~tns_ibc["name"].isin(exclude_names)].copy()
        tns_ibc = tns_ibc.rename(columns={"ra": "RA_deg", "declination": "DEC_deg"})
        return tns_ibc

    raise FileNotFoundError("Could not find a TNS table in this project.")


tns_ibc = load_tns_ibc()
tns_ibc[["name", "RA_deg", "DEC_deg"]]

# filter to only 2022jli, 2022esa, 2020sgf for testing
# tns_ibc = tns_ibc[tns_ibc["name"].isin(["2022jli", "2022esa", "2020sgf"])].copy()


/var/folders/9v/9mgg2nhs31x_zn1r4181_xym0000gn/T/ipykernel_95016/125804419.py:15: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  utc=True).dt.to_pydatetime()).jd > ZTF_FP_MIN_JD].copy()


,name,RA_deg,DEC_deg
53,2026ons,210.863596,-11.836359
190,2026olw,342.445013,-26.403500
395,2026gzf,149.928635,0.418438
742,2026naz,203.083744,1.570124
790,2026oao,187.577012,47.209222
...,...,...,...
182197,2018bvi,213.576627,10.678200
182526,2018bfu,198.870788,-13.325656
182783,2018avy,178.754792,32.075350
182931,2018aqf,276.703583,51.140469


### Create batch ZTF FP requests per year

In [ ]:
# Build and optionally submit ZTF Forced Photometry batch requests by discovery year.
# Each year gets one shared JD range, padded before Jan 1 and after Dec 31.
# Check status https://ztfweb.ipac.caltech.edu/cgi-bin/getBatchForcedPhotometryRequests.cgi
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from astropy.time import Time
from tqdm.auto import tqdm

ZTF_BATCH_FP_URL = "https://ztfweb.ipac.caltech.edu/cgi-bin/batchfp.py/submit"
ZTF_FP_REQUEST_DIR = Path("ztf_fp_requests")
ZTF_FP_REQUEST_DIR.mkdir(exist_ok=True)

PRE_DAYS = 50
POST_DAYS = 300
MAX_POSITIONS_PER_REQUEST = 1500
SLEEP_SECONDS_BETWEEN_SUBMISSIONS = 2.0
DRY_RUN = False


def year_jd_range(year, pre_days=PRE_DAYS, post_days=POST_DAYS):
    jd_start = Time(f"{year}-01-01 00:00:00", scale="utc").jd - pre_days
    jd_end = Time(f"{year + 1}-01-01 00:00:00", scale="utc").jd + post_days

    if jd_start < ZTF_FP_MIN_JD:
        jd_start = ZTF_FP_MIN_JD
    
    return jd_start, jd_end


def prepare_tns_ibc_year_batches(tns_ibc):
    targets = tns_ibc.copy()
    targets["discovery_time"] = pd.to_datetime(targets["discoverydate"], utc=True, errors="coerce")
    targets = targets[targets["discovery_time"].notna()].copy()
    targets["discovery_year"] = targets["discovery_time"].dt.year.astype(int)

    needed = ["name", "RA_deg", "DEC_deg", "discoverydate", "discovery_year"]
    missing = [col for col in needed if col not in targets.columns]
    if missing:
        raise KeyError(f"tns_ibc is missing columns: {missing}")

    return targets.sort_values(["discovery_year", "name"]).reset_index(drop=True)


def write_year_batch_files(targets_for_year, year, jd_start, jd_end):
    coord_path = ZTF_FP_REQUEST_DIR / f"tns_ibc_{year}_positions.txt"
    payload_path = ZTF_FP_REQUEST_DIR / f"tns_ibc_{year}_payload.json"

    lines = ["# name ra dec discoverydate\n"]
    for row in targets_for_year.itertuples(index=False):
        lines.append(
            f"{row.name} {float(row.RA_deg):.7f} {float(row.DEC_deg):.7f} {row.discoverydate}\n"
        )
    coord_path.write_text("".join(lines))

    payload_preview = {
        "year": int(year),
        "n_targets": int(len(targets_for_year)),
        "jdstart": float(jd_start),
        "jdend": float(jd_end),
        "coord_file": str(coord_path),
        "pre_days": PRE_DAYS,
        "post_days": POST_DAYS,
    }
    payload_path.write_text(json.dumps(payload_preview, indent=2))

    return coord_path, payload_path


def submit_batchfp_request(targets_for_year, jd_start, jd_end, email, userpass):
    # ZTF batch forced-photometry endpoint expects JSON-encoded arrays/values.
    ra_list = [float(f"{ra:.7f}") for ra in targets_for_year["RA_deg"]]
    dec_list = [float(f"{dec:.7f}") for dec in targets_for_year["DEC_deg"]]

    payload = {
        "ra": json.dumps(ra_list),
        "dec": json.dumps(dec_list),
        "jdstart": json.dumps(float(jd_start)),
        "jdend": json.dumps(float(jd_end)),
        "email": email,
        "userpass": userpass,
    }

    response = requests.post(
        ZTF_BATCH_FP_URL,
        auth=("ztffps", "dontgocrazy!"),
        data=payload,
        timeout=120,
    )
    return response


targets_batched = prepare_tns_ibc_year_batches(tns_ibc)
batch_summary_rows = []
submission_rows = []

# Use your existing helper if it has been defined in a previous cell.
email, userpass = get_ztffp_credentials()
print(f"Prepared {len(targets_batched)} targets in {targets_batched['discovery_year'].nunique()} discovery-year batches")

for year, year_targets in targets_batched.groupby("discovery_year", sort=True):
    
    jd_start, jd_end = year_jd_range(int(year))
    year_targets = year_targets.copy().reset_index(drop=True)

    if len(year_targets) > MAX_POSITIONS_PER_REQUEST:
        raise ValueError(
            f"Year {year} has {len(year_targets)} targets, more than "
            f"MAX_POSITIONS_PER_REQUEST={MAX_POSITIONS_PER_REQUEST}. Split this year further."
        )

    coord_path, payload_path = write_year_batch_files(year_targets, int(year), jd_start, jd_end)

    batch_summary_rows.append({
        "year": int(year),
        "n_targets": int(len(year_targets)),
        "jdstart": jd_start,
        "jdend": jd_end,
        "date_start_utc": Time(jd_start, format="jd", scale="utc").iso,
        "date_end_utc": Time(jd_end, format="jd", scale="utc").iso,
        "coord_file": str(coord_path),
        "payload_file": str(payload_path),
    })

    print(
        f"{year}: {len(year_targets)} targets, "
        f"JD {jd_start:.2f} .. {jd_end:.2f}, wrote {coord_path}"
    )

    if not DRY_RUN:
        if not email or not userpass:
            raise RuntimeError("Missing ZTF forced-photometry email/userpass credentials.")
        response = submit_batchfp_request(year_targets, jd_start, jd_end, email, userpass)
        submission_rows.append({
            "year": int(year),
            "status_code": int(response.status_code),
            "response_text": response.text[:2000],
        })
        print(f"  submitted: HTTP {response.status_code}")
        time.sleep(SLEEP_SECONDS_BETWEEN_SUBMISSIONS)

batch_summary = pd.DataFrame(batch_summary_rows)
batch_summary_path = ZTF_FP_REQUEST_DIR / "tns_ibc_year_batch_summary.csv"
batch_summary.to_csv(batch_summary_path, index=False)
print(f"Wrote {batch_summary_path}")

display(batch_summary)

if submission_rows:
    submission_summary = pd.DataFrame(submission_rows)
    submission_summary_path = ZTF_FP_REQUEST_DIR / "tns_ibc_year_batch_submissions.csv"
    submission_summary.to_csv(submission_summary_path, index=False)
    display(submission_summary)
else:
    print("DRY_RUN=True: no requests submitted. Set DRY_RUN=False to submit.")

Prepared 769 targets in 9 discovery-year batches
2018: 58 targets, JD 2458194.50 .. 2458784.50, wrote ztf_fp_requests/tns_ibc_2018_positions.txt
  submitted: HTTP 200
Wrote ztf_fp_requests/tns_ibc_year_batch_summary.csv


,year,n_targets,jdstart,jdend,date_start_utc,date_end_utc,coord_file,payload_file
0,2018,58,2458194.5,2458784.5,2018-03-17 00:00:00.000,2019-10-28 00:00:00.000,ztf_fp_requests/tns_ibc_2018_positions.txt,ztf_fp_requests/tns_ibc_2018_payload.json


,year,status_code,response_text
0,2018,200,Got: jdstart=2458194.5 jdend=2458784.5 email=a...
